# 🛒 LG Aimers — 온라인 채널 제품 판매량 예측 프로젝트 (최종본)

---

## 1. 프로젝트 개요 및 모델링 계획

### 1-1. 왜 이 모델링이 필요한가?

LG는 온라인 헬스케어 쇼핑몰에서 **15,890개의 제품**을 판매하고 있다. 제품 재고 관리, 마케팅 예산 배분, 프로모션 기획 등은 모두 **미래 판매량 예측의 정확도**에 크게 의존한다.

- **재고 부족 시**: 판매 기회 손실, 고객 이탈
- **재고 과잉 시**: 보관 비용 증가, 폐기 손실
- **브랜드 키워드 검색량** 같은 외부 신호를 활용하면 기존 룰 기반 예측보다 훨씬 정확한 수요 예측이 가능

이 프로젝트는 딥러닝 기반 시계열 모델(LSTM)을 활용하여 **다변량 패턴, 비선형 계절성, 이상치**를 처리하고 정확한 수요를 예측하는 것을 목표로 한다.

---

### 1-2. 타겟값 (What to Predict)

| 항목 | 내용 |
|------|------|
| **예측 대상** | 각 제품의 **일별 판매 수량** |
| **예측 기간** | `2023-04-05 ~ 2023-04-25` (총 **21일**) |
| **예측 단위** | 제품(Product) × 날짜(Date) |
| **총 예측 셀 수** | 15,890개 제품 × 21일 = **333,690개** |
| **타겟 값 범위** | 0 이상의 정수 (비음수 카운트 데이터) |

---

### 1-3. 데이터 구조 파악

- **train.csv**: 제품 정보 및 일별 판매 수량 (2022-01-01 ~ 2023-04-04)
- **sales.csv**: 동일 기간의 일별 매출액 (단가 변동 피처 생성에 활용)
- **product_info.csv**: 제품별 상세 특성 텍스트
- **brand_keyword_cnt.csv**: 일별 브랜드 키워드 검색 수

---

### 1-4. 핵심 도전 과제

- **Sparsity (희소성)**: 판매량의 약 64%가 0인 극심한 희소 데이터.
- **Scale Difference**: 제품별 판매량 스케일 차이가 매우 큼 (Log 변환 필수).
- **Multi-step Forecasting**: 21일이라는 긴 기간을 한 번에 예측해야 함.

---

### 1-5. 분석 파이프라인

| 단계 | 내용 |
|------|------|
| **STEP 0** | 라이브러리 불러오기 & 데이터 로드 |
| **STEP 1** | EDA — 데이터 탐색 및 시각화 |
| **STEP 2** | 데이터 전처리 & 피처 엔지니어링 (Lag, Rolling, Date, Keyword) |
| **STEP 3** | 딥러닝 모델 구성 (PyTorch LSTM) |
| **STEP 4** | 컴파일 (MAE Loss, Adam Optimizer) |
| **STEP 5** | Callbacks 정의 (EarlyStopping, ReduceLR) |
| **STEP 6** | 모델 학습 및 검증 |
| **STEP 7** | 학습 결과 및 예측 시각화 |
| **STEP 8** | 최종 예측 및 제출 파일 생성 |

## STEP 0. 라이브러리 불러오기 & 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import warnings
import os

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 재현성을 위한 시드 고정
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용할 장치: {device}")

# 데이터 로드
DATA_PATH = './'
print("📂 데이터 로딩 중...")
train = pd.read_csv(DATA_PATH + 'train.csv')
sales = pd.read_csv(DATA_PATH + 'sales.csv')
product_info = pd.read_csv(DATA_PATH + 'product_info.csv')
brand_kw = pd.read_csv(DATA_PATH + 'brand_keyword_cnt.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

date_cols = [col for col in train.columns if '202' in col]
pred_cols = [col for col in submission.columns if '202' in col]

print(f"✅ 데이터 로드 완료!")
print(f"   - 제품 수: {len(train):,}개")
print(f"   - 학습 기간: {len(date_cols)}일 ({date_cols[0]} ~ {date_cols[-1]})")
print(f"   - 예측 기간: {len(pred_cols)}일 ({pred_cols[0]} ~ {pred_cols[-1]})")

## STEP 1. EDA (탐색적 데이터 분석)

In [ ]:
# 1-1. 데이터 기본 진단
n_products = len(train)
n_days = len(date_cols)
zero_sales = (train[date_cols] == 0).sum().sum()
total_cells = n_products * n_days

print(f"--- [데이터 진단 보고서] ---")
print(f"✅ 분석할 제품 수: {n_products:,}개")
print(f"✅ 학습할 기간: {n_days}일")
print(f"✅ 카테고리 종류: {train['대분류'].unique()}")
print(f"⚠️ 판매량 0인 비중: {zero_sales / total_cells * 100:.2f}%")

# 1-2. 판매량 분포 시각화 (로그 변환 전후)
all_sales = train[date_cols].values.flatten()
nonzero_sales = all_sales[all_sales > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(nonzero_sales, bins=50, color='steelblue', alpha=0.8)
axes[0].set_title('원래 판매량 분포 (Log Scale Y)')
axes[0].set_yscale('log')

axes[1].hist(np.log1p(nonzero_sales), bins=50, color='coral', alpha=0.8)
axes[1].set_title('log1p 변환 후 판매량 분포')
plt.tight_layout()
plt.show()

In [ ]:
# 1-3. 시계열 패턴 분석 (요일별/월별)
dates = pd.to_datetime(date_cols)
daily_avg = train[date_cols].mean(axis=0)
daily_avg.index = dates

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 전체 트렌드
axes[0, 0].plot(dates, daily_avg.values, alpha=0.5)
axes[0, 0].plot(dates, daily_avg.rolling(7).mean(), color='red', label='7D Moving Avg')
axes[0, 0].set_title('Daily Avg Sales Trend')

# 요일별 패턴
dow_avg = daily_avg.groupby(dates.dayofweek).mean()
axes[0, 1].bar(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], dow_avg.values, color='steelblue')
axes[0, 1].set_title('Avg Sales by Day of Week')

# 검색량 vs 판매량 상관관계 (Log Scale)
brand_sales = train.groupby('브랜드')[date_cols].sum().sum(axis=1).reset_index(name='total_sales')
brand_kw_numeric = brand_kw.copy()
brand_kw_numeric[date_cols] = brand_kw_numeric[date_cols].apply(pd.to_numeric, errors='coerce')
brand_kw_total = brand_kw_numeric[date_cols].sum(axis=1).reset_index(name='total_kw')
brand_kw_total['브랜드'] = brand_kw['브랜드']

brand_analysis = brand_sales.merge(brand_kw_total, on='브랜드')
axes[1, 0].scatter(np.log1p(brand_analysis['total_kw']), np.log1p(brand_analysis['total_sales']), alpha=0.3, color='green')
axes[1, 0].set_title('Keyword Search vs Sales (Log-Log)')
axes[1, 0].set_xlabel('Log Search Count')
axes[1, 0].set_ylabel('Log Sales Count')

plt.tight_layout()
plt.show()

## STEP 2. 데이터 전처리 & 피처 엔지니어링

In [ ]:
print("🔄 Wide → Long 변환 중...")
train_long = train.melt(
    id_vars=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'],
    value_vars=date_cols, var_name='date', value_name='sales'
)
train_long['date'] = pd.to_datetime(train_long['date'])
train_long = train_long.sort_values(['ID', 'date']).reset_index(drop=True)

# 단가 피처
sales_long = sales.melt(id_vars=['ID'], value_vars=date_cols, var_name='date', value_name='revenue')
train_long['revenue'] = sales_long['revenue'].values
train_long['unit_price'] = train_long['revenue'] / (train_long['sales'] + 1)

# 로그 변환
train_long['sales_log'] = np.log1p(train_long['sales'])

print("📐 Lag & Rolling 피처 생성 중...")
for lag in [1, 7, 14, 21, 28]:
    train_long[f'lag_{lag}'] = train_long.groupby('ID')['sales_log'].shift(lag)

for window in [7, 14, 28]:
    train_long[f'rolling_mean_{window}'] = train_long.groupby('ID')['sales_log'].transform(lambda x: x.shift(1).rolling(window).mean())
    train_long[f'rolling_std_{window}'] = train_long.groupby('ID')['sales_log'].transform(lambda x: x.shift(1).rolling(window).std())

# 날짜 피처
train_long['day_of_week'] = train_long['date'].dt.dayofweek
train_long['month'] = train_long['date'].dt.month
train_long['is_weekend'] = (train_long['day_of_week'] >= 5).astype(int)
train_long['days_since_start'] = train_long.groupby('ID')['date'].transform(lambda x: (x - x.min()).dt.days)

print("🔢 카테고리 인코딩 중...")
label_encoders = {}
for col in ['대분류', '중분류', '소분류', '브랜드']:
    le = LabelEncoder()
    train_long[f'{col}_enc'] = le.fit_transform(train_long[col].astype(str))
    label_encoders[col] = le

train_long = train_long.fillna(0)
print("✅ 전처리 완료!")

## STEP 3. PyTorch Dataset & LSTM 모델 구성

In [ ]:
NUMERIC_FEATURES = (
    [f'lag_{l}' for l in [1, 7, 14, 21, 28]] +
    [f'rolling_mean_{w}' for w in [7, 14, 28]] +
    [f'rolling_std_{w}' for w in [7, 14, 28]] +
    ['day_of_week', 'month', 'is_weekend', 'days_since_start']
)
CAT_FEATURES = ['대분류_enc', '중분류_enc', '소분류_enc', '브랜드_enc']
N_CAT = {c: train_long[c].max() + 1 for c in CAT_FEATURES}

SEQ_LEN = 60
PRED_LEN = 21

class SalesDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, pred_len=PRED_LEN):
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.product_ids = df['ID'].unique()
        self.product_data = {pid: df[df['ID'] == pid].sort_values('date') for pid in self.product_ids}
        self.samples = [(pid, start_idx) for pid in self.product_ids 
                        for start_idx in range(len(self.product_data[pid]) - seq_len - pred_len + 1)]
    
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        pid, start_idx = self.samples[idx]
        data = self.product_data[pid]
        x_num = torch.tensor(data.iloc[start_idx:start_idx+self.seq_len][NUMERIC_FEATURES].values.astype(np.float32))
        x_cat = torch.tensor(data.iloc[0][CAT_FEATURES].values.astype(np.int64))
        y = torch.tensor(data.iloc[start_idx+self.seq_len:start_idx+self.seq_len+self.pred_len]['sales_log'].values.astype(np.float32))
        return x_num, x_cat, y

# 모델 정의
class SalesLSTM(nn.Module):
    def __init__(self, input_dim, cat_dims, emb_dims=[4, 8, 16, 32], hidden_dim=256, num_layers=2):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(n, d) for n, d in zip(cat_dims, emb_dims)])
        self.lstm = nn.LSTM(input_dim + sum(emb_dims), hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Linear(128, PRED_LEN))
        
    def forward(self, x_num, x_cat):
        embs = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=-1)
        embs = embs.unsqueeze(1).repeat(1, x_num.size(1), 1)
        x = torch.cat([x_num, embs], dim=-1)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

print("🧠 모델 및 데이터셋 구성 완료!")

## STEP 4-6. 학습 및 검증

In [ ]:
# Train/Valid Split
cutoff = pd.Timestamp('2023-03-14')
train_ds = SalesDataset(train_long[train_long['date'] <= cutoff])
valid_ds = SalesDataset(train_long[train_long['date'] > cutoff - pd.Timedelta(days=SEQ_LEN)])

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=512, shuffle=False)

model = SalesLSTM(len(NUMERIC_FEATURES), [N_CAT[c] for c in CAT_FEATURES]).to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("🚀 학습 시작...")
best_loss = float('inf')
for epoch in range(10): # 예시를 위해 10 에폭만 설정
    model.train()
    t_loss = 0
    for xn, xc, y in train_loader:
        xn, xc, y = xn.to(device), xc.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(xn, xc)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for xn, xc, y in valid_loader:
            xn, xc, y = xn.to(device), xc.to(device), y.to(device)
            v_loss += criterion(model(xn, xc), y).item()
    
    print(f"Epoch {epoch+1} | Train Loss: {t_loss/len(train_loader):.4f} | Val Loss: {v_loss/len(valid_loader):.4f}")
    if v_loss < best_loss: 
        best_loss = v_loss
        torch.save(model.state_dict(), 'best_model.pth')

## STEP 8. 최종 예측 및 제출 파일 생성

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
predictions = {}

print("🔮 최종 예측 진행 중...")
with torch.no_grad():
    for pid in train['ID'].unique():
        p_data = train_long[train_long['ID'] == pid].tail(SEQ_LEN)
        xn = torch.tensor(p_data[NUMERIC_FEATURES].values.astype(np.float32)).unsqueeze(0).to(device)
        xc = torch.tensor(p_data[CAT_FEATURES].iloc[0].values.astype(np.int64)).unsqueeze(0).to(device)
        
        pred_log = model(xn, xc).squeeze().cpu().numpy()
        pred_orig = np.round(np.clip(np.expm1(pred_log), 0, None)).astype(int)
        predictions[pid] = pred_orig

for idx, row in submission.iterrows():
    submission.loc[idx, pred_cols] = predictions.get(row['ID'], 0)

submission.to_csv('submission.csv', index=False)
print("✅ submission.csv 저장 완료! 프로젝트가 성공적으로 마무리되었습니다.")